In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA02 - Travel Lodging Entertainment (Public Officials)
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. DATA SOURCE: Uses COUPA FILES ONLY (7 files unioned).
   3. PUBLIC OFFICIAL FILTER: Filters strictly for rows where `PublicOfficial` = 'Y' or 'YES'.
   4. ACCOUNT PARSING: Extracts Cost Center and Category Code directly from the Account string.
   5. TARGET CATEGORY CODES: Hardcoded explicitly to the 29 target codes.
   6. CC MAPPING & THE 793 RULE: Maps to AU_IDs using the bootstrap. Applies the 
      exception rule that Category Code 793 is ONLY valid for AU 101016.
   7. FINAL OUTPUT: Calculates qualifying findings divided by total mapped source 
      records for the AU.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    SELECT 
        Account AS Raw_Account,
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cost_Center_ID,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Mapped_To_AU AS (
    SELECT 
        c.*,
        m.AU_ID
    FROM Coupa_Parsed c
    INNER JOIN CC_Mapping m 
        ON c.Cost_Center_ID = m.Mapped_CC
),

Flagged_AUs AS (
    SELECT 
        AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Mapped_To_AU
    WHERE UPPER(PublicOfficial) IN ('Y', 'YES')
      AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
      -- THE 793 EXCEPTION RULE
      AND NOT (CatCode = '793' AND AU_ID != '101016')
    GROUP BY AU_ID
),

Total_Records_By_AU AS (
    -- Count all mapped source records per AU for the denominator
    SELECT 
        AU_ID,
        COUNT(*) AS Total_Record_Count
    FROM Mapped_To_AU
    WHERE AU_ID IS NOT NULL
    GROUP BY AU_ID
)

SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA02' AS QuestionID,               
    CASE 
        WHEN COALESCE(r.Total_Record_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(f.Flagged_Count, 0) AS DOUBLE) / r.Total_Record_Count) * 100, 2) AS STRING) || '%'
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
LEFT JOIN Total_Records_By_AU r
    ON a.BusinessID = r.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA02 - AU Level Calculation Review
   PURPOSE: One row per AU showing how the record-based percentage response was
   calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    SELECT 
        Account AS Raw_Account,
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cost_Center_ID,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Mapped_To_AU AS (
    SELECT 
        c.*,
        m.AU_ID
    FROM Coupa_Parsed c
    LEFT JOIN CC_Mapping m 
        ON c.Cost_Center_ID = m.Mapped_CC
),

AU_Debug AS (
    SELECT 
        AU_ID,
        COUNT(*) AS Bridged_Row_Count,
        SUM(CASE WHEN UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES') THEN 1 ELSE 0 END) AS Public_Official_Row_Count,
        SUM(CASE WHEN CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892') THEN 1 ELSE 0 END) AS Valid_CatCode_Row_Count,
        SUM(CASE WHEN AU_ID IS NOT NULL
                  AND UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES')
                  AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
                  AND NOT (CatCode = '793' AND AU_ID != '101016')
                 THEN 1 ELSE 0 END) AS Qualifying_Row_Count,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN AU_ID IS NOT NULL
                                                      AND UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES')
                                                      AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
                                                      AND NOT (CatCode = '793' AND AU_ID != '101016')
                                                     THEN CatCode END))) AS Passing_CatCodes
    FROM Mapped_To_AU
    WHERE AU_ID IS NOT NULL
    GROUP BY AU_ID
)

SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EBA02' AS QuestionID,
    CASE 
        WHEN COALESCE(d.Bridged_Row_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(d.Qualifying_Row_Count, 0) AS DOUBLE) / d.Bridged_Row_Count) * 100, 2) AS STRING) || '%'
    END AS Response,
    COALESCE(d.Bridged_Row_Count, 0) AS Bridged_Row_Count,
    COALESCE(d.Public_Official_Row_Count, 0) AS Public_Official_Row_Count,
    COALESCE(d.Valid_CatCode_Row_Count, 0) AS Valid_CatCode_Row_Count,
    COALESCE(d.Qualifying_Row_Count, 0) AS Qualifying_Row_Count,
    COALESCE(d.Qualifying_Row_Count, 0) AS Numerator,
    COALESCE(d.Bridged_Row_Count, 0) AS Denominator,
    CASE 
        WHEN COALESCE(d.Bridged_Row_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(d.Qualifying_Row_Count, 0) AS DOUBLE) / d.Bridged_Row_Count) * 100, 2) AS STRING) || '%'
    END AS Calculated_Percentage,
    CASE 
        WHEN COALESCE(d.Bridged_Row_Count, 0) = 0 THEN CONCAT('Response = 0% because denominator = 0. Findings = ', CAST(COALESCE(d.Qualifying_Row_Count, 0) AS STRING), ' and this AU has no mapped source records.')
        ELSE CONCAT('Response = ', CAST(ROUND((CAST(COALESCE(d.Qualifying_Row_Count, 0) AS DOUBLE) / d.Bridged_Row_Count) * 100, 2) AS STRING), '% because findings ', CAST(COALESCE(d.Qualifying_Row_Count, 0) AS STRING), ' / total mapped records ', CAST(d.Bridged_Row_Count AS STRING), ' * 100. Passing CatCodes: ', COALESCE(d.Passing_CatCodes, 'n/a'))
    END AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN AU_Debug d
    ON mast.BusinessID = d.AU_ID
ORDER BY mast.BusinessID;